# 05: Transformer Model Specification

This notebook presents the self-supervised Transformer architecture for EGM analysis.

## Key Features
- **Self-Supervised Pre-training**: Learn representations without labels
- **Multiple Masking Strategies**: Cross-channel, intra-channel, autoregressive
- **Contrastive Learning**: NT-Xent loss for view consistency
- **Multi-Task Loss**: Reconstruction + contrastive + regularization

## Architecture
- Patch Embedding (50 timesteps → 256-dim embedding)
- Transformer Encoder (8 layers, 8 attention heads)
- Reconstruction Head (masked region prediction)
- Projection Head (contrastive learning)
- Classification Head (scar localization)

In [ ]:
# Setup
import os
import sys
import json
from pathlib import Path

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)
os.chdir(project_root)

from src import config, utils
from src.models import transformer

## 1. Model Architecture Specification

In [ ]:
# Create model specification
model_spec = transformer.create_egm_transformer_model(
    n_channels=3,
    n_timesteps=2500,
    patch_size=50,
    d_model=256,
    nhead=8,
    num_encoder_layers=8,
    dim_feedforward=1024,
    dropout=0.1,
    output_dim=3  # Scar classes
)

print("Transformer Model Specification:")
print(json.dumps({k: v for k, v in model_spec.items() if k != 'architecture'}, indent=2))

In [ ]:
# Detailed architecture
print("\n" + "="*60)
print("DETAILED ARCHITECTURE")
print("="*60)

print("\nInput:")
for key, val in model_spec['input_spec'].items():
    print(f"  {key}: {val}")

print("\nPatch Embedding:")
for key, val in model_spec['patch_embedding'].items():
    print(f"  {key}: {val}")

print("\nTransformer Encoder:")
for key, val in model_spec['transformer_encoder'].items():
    print(f"  {key}: {val}")

print("\nDecoder Heads:")
for head_name, head_spec in model_spec['decoder_heads'].items():
    print(f"\n  {head_name.upper()}:")
    for key, val in head_spec.items():
        if key != 'type' and key != 'description':
            print(f"    {key}: {val}")

## 2. Self-Supervised Learning Strategy

In [ ]:
print("\n" + "="*60)
print("MASKING STRATEGIES")
print("="*60)

masking_strategies = transformer.PretrainingDatasetSpec.get_masking_strategies()

for strategy_name, strategy_spec in masking_strategies.items():
    print(f"\n{strategy_name.upper()}:")
    print(f"  Description: {strategy_spec['description']}")
    print(f"  Probability: {strategy_spec['probability']:.2%}")
    if strategy_spec['parameters']:
        print(f"  Parameters:")
        for key, val in strategy_spec['parameters'].items():
            print(f"    - {key}: {val}")

In [ ]:
print("\n" + "="*60)
print("DATA AUGMENTATION STRATEGIES")
print("="*60)

augmentation_strategies = transformer.PretrainingDatasetSpec.get_augmentation_strategies()

for aug_name, aug_spec in augmentation_strategies.items():
    print(f"\n{aug_name.upper()}:")
    print(f"  Probability: {aug_spec['probability']:.2%}")
    for key, val in aug_spec.items():
        if key != 'probability':
            print(f"  {key}: {val}")

## 3. Loss Function Components

In [ ]:
print("\n" + "="*60)
print("LOSS WEIGHTS")
print("="*60)

loss_weights = transformer.LossWeights.get_all_weights()

print("\nTask Losses:")
print(f"  Cross-channel reconstruction: {loss_weights['cross_channel']:.2f}")
print(f"  Intra-channel reconstruction: {loss_weights['intra_channel']:.2f}")
print(f"  Autoregressive reconstruction: {loss_weights['autoregressive']:.2f}")
print(f"  Contrastive (NT-Xent): {loss_weights['contrastive']:.2f}")

print("\nRegularization Losses:")
print(f"  Smoothness (1st deriv): {loss_weights['smoothness']:.2f}")
print(f"  Curvature (2nd deriv): {loss_weights['curvature']:.2f}")
print(f"  MAE weight: {loss_weights['mae']:.2f}")
print(f"  STFT spectral loss: {loss_weights['stft']:.2f}")

total = sum(loss_weights.values())
print(f"\n  Total weight sum: {total:.2f}")

In [ ]:
# Reconstruction loss components
recon_loss_spec = transformer.get_reconstruction_loss_spec()

print("\nReconstruction Loss Components:")
for comp_name, comp_spec in recon_loss_spec['components'].items():
    print(f"\n  {comp_name.upper()}:")
    print(f"    Name: {comp_spec['name']}")
    if 'description' in comp_spec:
        print(f"    Description: {comp_spec['description']}")
    print(f"    Weight: {comp_spec['weight']:.2f}")

## 4. Training Configuration

In [ ]:
training_config = transformer.get_transformer_training_config()

print("\n" + "="*60)
print("TRAINING CONFIGURATION")
print("="*60)

for key, value in training_config.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

## 5. Contrastive Learning

In [ ]:
contrastive_spec = transformer.get_contrastive_loss_spec()

print("\n" + "="*60)
print("CONTRASTIVE LEARNING (NT-Xent)")
print("="*60)
print("\nPurpose: Learn consistent representations across different augmentations")
print("\nSpecification:")
for key, value in contrastive_spec.items():
    print(f"  {key}: {value}")

print("\nHow it works:")
print("  1. Generate two different augmented views of same EGM signal")
print("  2. Project both through the projection head")
print("  3. Maximize similarity: sim(view1, view2) high")
print("  4. Minimize similarity: sim(view1, other_samples) low")
print("  5. Temperature = 0.07 controls smoothness of similarity distribution")

## 6. Usage in PyTorch/TensorFlow

In [ ]:
implementation_code = """
# PyTorch-style pseudo-code for Transformer implementation:

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class EGMTransformer(nn.Module):
    def __init__(self, **config):
        super().__init__()
        
        # Patch embedding
        self.patch_embedding = nn.Linear(
            in_features=config['patch_size'] * config['n_channels'],
            out_features=config['d_model']
        )
        
        # Positional encoding (sinusoidal)
        self.positional_encoding = PositionalEncoding(config['d_model'])
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config['d_model'],
            nhead=config['nhead'],
            dim_feedforward=config['dim_feedforward'],
            dropout=config['dropout']
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=config['num_encoder_layers']
        )
        
        # Multiple heads
        self.reconstruction_head = nn.Linear(config['d_model'], config['patch_size'])
        self.projection_head = nn.Sequential(
            nn.Linear(config['d_model'], 512),
            nn.ReLU(),
            nn.Linear(512, 256)
        )
        self.classification_head = nn.Linear(config['d_model'], config['output_dim'])
    
    def forward(self, x_masked):
        # x_masked: (batch, n_channels, n_timesteps) with masking applied
        batch_size = x_masked.shape[0]
        
        # Patchify and embed
        x_patches = self.patchify(x_masked)  # (batch, n_patches, d_model)
        x_embedded = self.patch_embedding(x_patches)
        
        # Add positional encoding
        x_embedded = self.positional_encoding(x_embedded)
        
        # Transformer encoding
        x_encoded = self.transformer_encoder(x_embedded)
        
        # Multiple outputs
        reconstructed = self.reconstruction_head(x_encoded)
        projected = self.projection_head(x_encoded.mean(dim=1))  # Average pooling
        classification = self.classification_head(x_encoded.mean(dim=1))
        
        return reconstructed, projected, classification

# Training loop
model = EGMTransformer(**config)
optimizer = AdamW(model.parameters(), lr=config['learning_rate'])
scheduler = CosineAnnealingLR(optimizer, T_max=config['epochs'])

for epoch in range(config['epochs']):
    for batch in data_loader:
        # Apply masking strategies
        x_view1, mask1 = apply_masking_strategy(batch, strategy='cross_channel')
        x_view2, mask2 = apply_masking_strategy(batch, strategy='intra_channel')
        
        # Forward pass
        recon1, proj1, class1 = model(x_view1)
        recon2, proj2, class2 = model(x_view2)
        
        # Compute losses
        loss_recon = reconstruction_loss(recon1, batch, mask1)
        loss_contrastive = nt_xent_loss(proj1, proj2, temperature=0.07)
        loss_total = loss_recon + loss_contrastive
        
        # Backward pass
        optimizer.zero_grad()
        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    
    scheduler.step()
"""

print(implementation_code)

## 7. Pre-training vs Fine-tuning

**Pre-training Phase:**
- Train on large unlabeled EGM dataset
- Learn general patterns and features
- No label information needed
- Output: Pre-trained encoder weights

**Fine-tuning Phase:**
1. Load pre-trained encoder
2. Add classification head (if not already there)
3. Train with labeled scar tissue data
4. Lower learning rate, early stopping
5. Output: Scar localization model